# M3: direct CLV influence in LightGCN propagation

Dunnhumby seed-42 test-only protocol check. Former train and validation are merged through DAY 697, every arm trains for a fixed 100 epochs, and DAY 698--704 is evaluated once. Later rows are ignored. There is no validation selection, early stopping, or holdout evaluation.

## Setup

Mount Drive and check out the reviewed source commit. Required inputs are the Dunnhumby raw CSV files under `/content/drive/MyDrive/논문/data/raw/dunnhumby/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = '934c675f0325c8a262d6ab13de698489b2f43651'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('Pinned execution source:', actual_sha)

## Preflight checks

Confirm GPU availability, seed 42, fixed 100-epoch training, merged train+validation, test-only evaluation, and disabled holdout evaluation before training.

In [ ]:
import json, torch
from lightgcn_clv_m3_mass_preserving import (
    configure_m3_clv_influence_test_run,
    preflight_summary,
    run_test,
)

cfg = configure_m3_clv_influence_test_run()
assert torch.cuda.is_available(), 'Select a GPU runtime before running.'
summary = preflight_summary(cfg)
assert summary['seeds'] == [42]
assert summary['epochs'] == 100
assert summary['validation_constructed'] is False
assert summary['validation_selection'] is False
assert summary['early_stopping'] is False
assert summary['holdout_evaluation'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

## Run

Train M1 and the four fixed M3 arms for 100 epochs. Each completed arm is cached before its single test evaluation, so a reconnect does not evaluate the same completed arm again.

In [ ]:
result_df = run_test(cfg)

## Results and decision

Report the single-seed test values descriptively. Standard deviation and confidence intervals remain missing for one seed; no significance or model-selection claim is made.

In [ ]:
from IPython.display import display

columns = [
    'model_id', 'role',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'price_purchase_amount_weighted_hit@20',
    'price_purchase_amount_weighted_hit@50',
    'mean_recommended_price_percentile@10',
    'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
    'user_value_tendency_recommended_price_alignment',
]
available = [column for column in columns if column in result_df.columns]
test_rows = result_df[result_df['split'].eq('test')][available]
display(test_rows.sort_values('model_id'))
print('Seed summary (SD/CI are missing for the one-seed pilot):')
display(result_df.attrs['absolute_summary'])
print('Same-seed differences from M1:')
display(result_df.attrs['paired_summary'])
print('Result files:', result_df.attrs['result_paths'])